# Naive Bayes event models — A Comparison of Event Models for Naive Bayes Text Classification (1998)

_Papers · Classical ML_

**Showed that the two things everyone called "Naive Bayes" for text are different generative models, and that the word-count (multinomial) one usually wins.**

---

This notebook is a practice scaffold for the **Naive Bayes event models — A Comparison of Event Models for Naive Bayes Text Classification (1998)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, math
torch.set_printoptions(precision=6)

# --- 0. Toy corpus from the worked example. Vocab |V|=5: 0=fun 1=couple 2=love 3=fast 4=furious ---
# class 0 = romance, class 1 = action. Each row is a document's word-count vector N_it (bag of words).
X = torch.tensor([[2.,1,1,0,0],   # d0 "fun fun love couple"   -> class 0
                  [0.,1,2,0,0],   # d1 "couple love love"      -> class 0
                  [0.,0,0,1,2],   # d2 "fast furious furious"  -> class 1
                  [1.,0,0,2,1]])  # d3 "fast fast fun furious" -> class 1
y = torch.tensor([0, 0, 1, 1])
V = X.shape[1]; C = 2; alpha = 1.0           # |V|=5, 2 classes, Laplace add-one

# --- 1. Fit: log-prior (Eqn 4), Laplace-smoothed log word-probs (Eqn 6). Pure counting. ---
counts_c  = torch.tensor([(y == c).sum() for c in range(C)], dtype=torch.float)
log_prior = torch.log(counts_c / counts_c.sum())                       # Eqn 4 (hard labels)

word_counts = torch.stack([X[y == c].sum(0) for c in range(C)])        # [C, V] per-class word totals
num   = alpha + word_counts                                            # 1 + count   (numerator)
denom = alpha * V + word_counts.sum(1, keepdim=True)                   # |V| + total (denominator)
log_theta = torch.log(num / denom)                                     # Eqn 6: log P(w_t | c)

print("log_prior     :", [round(v, 4) for v in log_prior.tolist()])
print("log_theta c0  :", [round(v, 4) for v in log_theta[0].tolist()])
print("log_theta c1  :", [round(v, 4) for v in log_theta[1].tolist()])

# --- 2. Classify (Eqn 7 argmax): score = log_prior + N . log_theta, then take the largest. ---
test = torch.tensor([[1., 0, 2, 0, 0]])      # "love love fun" = 1x fun(0) + 2x love(2)  -> expect class 0
log_post = log_prior + test @ log_theta.T    # [1, C]; the shared P(d) cancels in the argmax
pred = log_post.argmax(1)
print("log_post      :", [round(v, 4) for v in log_post[0].tolist()])
print("prediction    :", pred.item(), "(0 = romance, expected 0)")

# --- 3. VERIFY against an independent hand-computed reference (the worked example). ---
# class 0 totals d0+d1=[2,2,3,0,0] sum 7 -> denom 5+7=12 ;  class 1 totals d2+d3=[1,0,0,3,3] sum 7 -> 12
ref_theta = torch.tensor([[(1+w)/12 for w in [2,2,3,0,0]],
                          [(1+w)/12 for w in [1,0,0,3,3]]])
ref_logpost = torch.tensor([[math.log(0.5) + 1*math.log(ref_theta[0,0]) + 2*math.log(ref_theta[0,2]),
                             math.log(0.5) + 1*math.log(ref_theta[1,0]) + 2*math.log(ref_theta[1,2])]])
assert torch.allclose(log_theta, ref_theta.log(), atol=1e-6), "log_theta != hand reference"
assert torch.allclose(log_post,  ref_logpost,     atol=1e-6), "log_post  != hand reference"
assert pred.item() == 0
print("allclose vs HAND reference :", True)

# --- 4. VERIFY our tensors ARE scikit-learn's MultinomialNB (normalize with logsumexp first). ---
from sklearn.naive_bayes import MultinomialNB
sk = MultinomialNB(alpha=1.0).fit(X.numpy(), y.numpy())
sk_logproba = torch.tensor(sk.predict_log_proba(test.numpy()), dtype=torch.float)
my_logproba = (log_post - torch.logsumexp(log_post, 1, keepdim=True)).float()
print("sklearn log-proba          :", [round(v, 6) for v in sk_logproba[0].tolist()])
print("ours    log-proba          :", [round(v, 6) for v in my_logproba[0].tolist()])
print("allclose vs sklearn        :", torch.allclose(my_logproba, sk_logproba, atol=1e-5))

# Our small run -> log_theta c0 [-1.3863,-1.3863,-1.0986,-2.4849,-2.4849];
#   log_post [-4.2767,-7.4547]; prediction 0; both allclose checks True.
# These are OUR numbers on a 4-document toy corpus, not the paper's reported accuracies.

## Visualize the data & results

_On a toy text corpus, how do the multinomial and multivariate Bernoulli Naive Bayes event models compare as the vocabulary grows?_

In [ ]:
import numpy as np
# Reproduce the paper's qualitative trend (Figs 1-3): multinomial holds up as the
# vocabulary grows; multivariate Bernoulli (presence/absence only) falls off.
def make_corpus(n_per_class=120, doc_len=40, vocab=400, seed=0):
    rng = np.random.default_rng(seed)
    p0 = np.ones(vocab); p1 = np.ones(vocab)
    p0[:5] += 12; p0[5:10] += 1            # class 0 favors words 0-4
    p1[5:10] += 12; p1[:5] += 1            # class 1 favors words 5-9 (rest is shared noise)
    p0 /= p0.sum(); p1 /= p1.sum()
    X, y = [], []
    for c, p in [(0, p0), (1, p1)]:
        for _ in range(n_per_class):
            X.append(np.bincount(rng.choice(vocab, doc_len, p=p), minlength=vocab)); y.append(c)
    return np.array(X, float), np.array(y)

def split(X, y, frac=0.5, seed=1):
    r = np.random.default_rng(seed); idx = r.permutation(len(y)); k = int(len(y)*frac)
    return X[idx[:k]], y[idx[:k]], X[idx[k:]], y[idx[k:]]

def multinomial(Xtr, ytr, Xte, V, a=1.0):     # Eqns 4-6, classify by argmax of log-score
    wc = np.stack([Xtr[ytr==c].sum(0) for c in range(2)])
    lt = np.log((a+wc)/(a*V + wc.sum(1, keepdims=True)))
    lp = np.log([(ytr==c).mean() for c in range(2)])
    return (lp + Xte @ lt.T).argmax(1)

def bernoulli(Xtr, ytr, Xte, V, a=1.0):        # Eqns 2-3: presence/absence only
    B = (Xtr>0).astype(float)
    th = np.stack([(a + B[ytr==c].sum(0))/(2*a + (ytr==c).sum()) for c in range(2)])
    lp = np.log([(ytr==c).mean() for c in range(2)]); Bte = (Xte>0).astype(float)
    sc = np.stack([lp[c] + (Bte*np.log(th[c]) + (1-Bte)*np.log(1-th[c])).sum(1) for c in range(2)], 1)
    return sc.argmax(1)

X, y = make_corpus(seed=0); Xtr, ytr, Xte, yte = split(X, y)
order = np.argsort(-X.sum(0))                  # rank words by frequency
for K in [10, 20, 40, 80, 160, 320]:
    keep = order[:K]; xtr, xte = Xtr[:, keep], Xte[:, keep]
    mn = (multinomial(xtr, ytr, xte, K) == yte).mean()
    bn = (bernoulli(xtr, ytr, xte, K) == yte).mean()
    print(f"V={K:4d}  multinomial={mn:.4f}  bernoulli={bn:.4f}")
# Our small run ->
# V=  10  multinomial=0.9917  bernoulli=0.9917
# V=  20  multinomial=0.9917  bernoulli=0.9833
# V=  40  multinomial=0.9750  bernoulli=0.9750
# V=  80  multinomial=0.9750  bernoulli=0.9667
# V= 160  multinomial=0.9583  bernoulli=0.9417
# V= 320  multinomial=0.9583  bernoulli=0.8500

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The event-model ablation. Take the multinomial Naive Bayes you built and swap only its
            document representation for the multivariate Bernoulli one: replace counts $N_{it}$ with presence
            flags $B_{it}=\mathbb{1}[N_{it}\gt 0]$ and score with the Bernoulli likelihood (Eqn 2, which also
            multiplies in a $(1-\theta)$ factor for every absent word). Train both on the same toy corpus and
            sweep the vocabulary size from small to large. What do you expect, and what does the result
            demonstrate about the paper's claim?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Fit the multinomial (Eqns 4-6) and the Bernoulli (Eqns 3-4, score by Eqn 2) on the identical training split. — _Holding data and split fixed isolates the event model as the only variable._
- Sweep vocabulary by keeping the top-K most frequent words, K small to large; record test accuracy for each model. — _The paper's Figs 1-3 vary vocabulary size on the x-axis; the divergence appears as the vocabulary grows._
- Observe the two are close at small K, but the Bernoulli accuracy drops as K grows while the multinomial stays high. — _Adding many rare words gives the Bernoulli model many noisy presence/absence factors and it cannot use word frequency; the multinomial's counts stay informative._

**Answer:** At a small vocabulary the two event models are about equal (sometimes Bernoulli is even
                 slightly ahead). As the vocabulary grows, the multivariate Bernoulli accuracy falls off
                 while the multinomial stays high &mdash; exactly the paper's finding (Abstract: Bernoulli
                 "performs well with small vocabulary sizes, but &hellip; the multinomial performs even better at
                 larger vocabulary sizes," for a 27% average error reduction). The CODEVIZ panel shows this
                 crossover on our toy corpus. The reason: the multinomial uses word frequency, which stays
                 informative; the Bernoulli only sees presence/absence and is swamped by the many rare words.

</details>

**Problem 2.** Why does Laplace (add-one) smoothing have $|V|$ in the denominator of Eqn (6), rather than $+1$ or
            $+|\mathcal{C}|$ (the number of classes)? Show that without $|V|$ the per-class word probabilities
            would not sum to 1.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write the unsmoothed estimate $\hat\theta_{w_t\mid c}=N_{\cdot t}/\sum_s N_{\cdot s}$ and confirm $\sum_t\hat\theta=1$. — _A word distribution must sum to 1 over the vocabulary._
- Add $\alpha$ to each of the $|V|$ numerators; the total added mass is $\alpha|V|$. — _Smoothing every word adds $\alpha$ exactly $|V|$ times._
- To keep the sum at 1, add the same $\alpha|V|$ to the denominator: $\hat\theta=(\alpha+N_{\cdot t})/(\alpha|V|+\sum_s N_{\cdot s})$. — _Numerator and denominator must add the same total mass for normalization to hold._

**Answer:** Summing Eqn (6) over all words: $\sum_{t}\frac{\alpha+N_{\cdot t}}{\alpha|V|+\sum_s N_{\cdot s}}
                 =\frac{\alpha|V|+\sum_t N_{\cdot t}}{\alpha|V|+\sum_s N_{\cdot s}}=1$. Because we added the
                 pseudo-count $\alpha$ to each of the $|V|$ words, we injected $\alpha|V|$ of extra mass into the
                 numerators, so the denominator must absorb exactly $\alpha|V|$ for the distribution to stay
                 normalized. Using $+1$ or $+|\mathcal{C}|$ instead would make the per-class probabilities not
                 sum to 1. With $\alpha=1$ this is the paper's Eqn (6).

</details>

**Problem 3.** Using the worked-example parameters (class 0 word probs $[0.25,0.25,0.3333,0.0833,0.0833]$, class 1
            $[0.1667,0.0833,0.0833,0.3333,0.3333]$, equal priors $0.5$), classify the test document
            "fast furious" $=[0,0,0,1,1]$ by hand. Give the log-score for each class and the
            prediction.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- class 0 score $=\log 0.5 + 1\cdot\log 0.0833 + 1\cdot\log 0.0833 = -0.6931 -2.4849 -2.4849 = -5.6629$. — _"fast" is index 3 and "furious" is index 4; both have prob 0.0833 in class 0._
- class 1 score $=\log 0.5 + 1\cdot\log 0.3333 + 1\cdot\log 0.3333 = -0.6931 -1.0986 -1.0986 = -2.8903$. — _Both words have prob 0.3333 in class 1 (action)._
- Compare: $-2.8903 \gt -5.6629$, so predict class 1. — _The larger (less negative) log-score wins the argmax (Eqn 7)._

**Answer:** class 0 score $=-5.6629$, class 1 score $=-2.8903$. Since $-2.8903 \gt -5.6629$, the model
                 predicts class 1 (action) &mdash; correct, because "fast" and "furious" are both high-
                 probability action words. This is the same $\arg\max$ rule (Eqn 7) the notebook computes; the
                 shared $P(d)$ denominator cancels, so we only need prior $\times$ likelihood in log space.

</details>